# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamikshaBurte/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup & Repository Initialization for Colab
import os, sys, subprocess

REPO_URL = "https://github.com/SamikshaBurte/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(f"/content/{REPO_DIR}")

print("Working Directory:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Dataset not found — check folder structure."
print("Repo setup complete! Dataset loaded.")

Working Directory: /content/flyrank-ml-internship
Repo setup complete! Dataset loaded.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load raw dataset and prepare target and features
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['target'] = (df['trend_direction'] == 'down').astype(int)

features = ['impressions_90d', 'ctr', 'avg_position', 'content_age_days']
X = df[features].fillna(0)
y = df['target']

print(f"Dataset Shape : {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Target Baseline Rate (Declining) : {y.mean() * 100:.2f}%")

Dataset Shape : 30,000 rows x 4 features
Target Baseline Rate (Declining) : 54.21%


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

# Train/Test split reserving 20% holdout set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("=== Split Verification ===")
print(f"Train Set Size : {len(X_train):,} rows ({y_train.mean()*100:.2f}% positive)")
print(f"Test Set Size  : {len(X_test):,} rows ({y_test.mean()*100:.2f}% positive)")

=== Split Verification ===
Train Set Size : 24,000 rows (54.21% positive)
Test Set Size  : 6,000 rows (54.20% positive)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Evaluate Week-4 Heuristic Baseline on Test Set
test_df = X_test.copy()
test_df['target'] = y_test
test_df['baseline_score'] = (test_df['content_age_days'] / test_df['content_age_days'].max() * 0.4) + \
                            (test_df['impressions_90d'] / test_df['impressions_90d'].max() * 0.4) + \
                            ((1 - test_df['ctr']) * 0.2)

top_50_base = test_df.sort_values(by='baseline_score', ascending=False).head(50)
base_p50 = top_50_base['target'].mean()

# 2. Train Models
lr_model = LogisticRegression(random_state=42).fit(X_train, y_train)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_train, y_train)

# 3. Predict Probabilities on Test Set
lr_probs = lr_model.predict_proba(X_test)[:, 1]
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# Compute Precision@50
top_50_lr_idx = np.argsort(-lr_probs)[:50]
top_50_rf_idx = np.argsort(-rf_probs)[:50]

lr_p50 = y_test.iloc[top_50_lr_idx].mean()
rf_p50 = y_test.iloc[top_50_rf_idx].mean()

# 4. Display Summary Table
comparison_table = pd.DataFrame({
    'Approach': ['Week 4 Rule Baseline', 'Logistic Regression', 'Random Forest (Capstone Model)'],
    'Precision@50': [f"{base_p50*100:.2f}%", f"{lr_p50*100:.2f}%", f"{rf_p50*100:.2f}%"],
    'ROC-AUC Score': ['N/A', f"{roc_auc_score(y_test, lr_probs):.4f}", f"{roc_auc_score(y_test, rf_probs):.4f}"]
})

print("=== Model vs Baseline Performance Comparison ===")
print(comparison_table.to_string(index=False))

=== Model vs Baseline Performance Comparison ===
                      Approach Precision@50 ROC-AUC Score
          Week 4 Rule Baseline       56.00%           N/A
           Logistic Regression       60.00%        0.5953
Random Forest (Capstone Model)       82.00%        0.7177


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature Importance Analysis
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

print("=== Random Forest Feature Importances ===")
print(importances.to_string())

# Error Inspection on False Positives
X_test_analysis = X_test.copy()
X_test_analysis['actual'] = y_test
X_test_analysis['pred_prob'] = rf_probs
top_50_predictions = X_test_analysis.sort_values(by='pred_prob', ascending=False).head(50)

false_positives = top_50_predictions[top_50_predictions['actual'] == 0]
print(f"\nFalse Positives in Top 50 Predictions: {len(false_positives)} rows")
print("Sample False Positive Features:")
print(false_positives[['impressions_90d', 'ctr', 'avg_position', 'content_age_days']].head(3).to_string(index=False))

=== Random Forest Feature Importances ===
impressions_90d     0.456047
content_age_days    0.258214
avg_position        0.214217
ctr                 0.071522

False Positives in Top 50 Predictions: 9 rows
Sample False Positive Features:
 impressions_90d  ctr  avg_position  content_age_days
             284 0.00          21.3               165
             247 0.00          16.8               165
            3131 0.03          21.2               139


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.